# Fraud Detection Model Fintech MLOps Stack
## Notebook 2 Preprocessing

Stage 3 takes the cleaned dataset from Notebook 1 and prepares it for modelling scaling features, splitting into train and test sets.


By the end of this notebook, we have a fully preprocessed training set ready to feed into the Auto gluon ensemble in Notebook 3.

#### Note: This notebook covers Stages 3 of our fraud detection pipeline(check notebook 1 for reference of full pipeline).

## Imports and Setup

#### We import all libraries needed for preprocessing and feature engineering. We also reload the cleaned dataset saved from Notebook 1 EDA, specifically the version after duplicate removal.

In [7]:
# Core libraries 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# Display settings 
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

print("Libraries imported successfully.")

Libraries imported successfully.


## Preprocessing Pipeline


### Load and Prepare the Cleaned Dataset

#### We reload the dataset and reapply the duplicate removal step from Notebook 1. This keeps Notebook 2 fully self-contained it can be run independently without depending on any in-memory state from Notebook 1.

In [8]:
# Load the raw dataset 
df = pd.read_csv('../data/creditcard.csv')

# Remove duplicate rows 
# Duplicates can bias the anomaly score if a transaction appears
# multiple times, Isolation Forest treats the dense cluster as
# "more normal" than it actually is
df = df.drop_duplicates()

# Confirm dataset shape and class distribution 
total       = len(df)
fraud_count = df['Class'].sum()
legit_count = total - fraud_count
fraud_rate  = fraud_count / total * 100

print(f"Total transactions:     {total:,}")
print(f"Legitimate:             {legit_count:,}")
print(f"Fraud:                  {fraud_count:,}")
print(f"Fraud rate:             {fraud_rate:.4f}%")
print(f"Features:               {df.shape[1] - 1} (excluding Class)")
print(f"Columns:                {df.columns.tolist()}")

Total transactions:     283,726
Legitimate:             283,253
Fraud:                  473
Fraud rate:             0.1667%
Features:               30 (excluding Class)
Columns:                ['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class']


##  Train/Test Split (Data Leakage Protection)

#### We split before any scaling or model fitting. The test set must remain completely untouched throughout the entire preprocessing pipeline. This is the fundamental rule of preventing data leakage.We use stratify=y to preserve the 0.1667% fraud rate in both splits so the test set reflects the real-world class distribution.

In [9]:
# Separate features from target 
# We keep all 30 features no dropping, no selection
X = df.drop('Class', axis=1)   # shape: (283726, 30)
y = df['Class']                 # 0 = legitimate, 1 = fraud

#  Stratified 80/20 split 
# stratify=y ensures both splits preserve the 0.1667% fraud rate
# random_state=42 guarantees identical split every time code runs
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,  # 70% train, 30% test
    random_state=0, # reproducibility
    stratify=y
)

# Confirm split sizes and fraud rates 
train_fraud_rate = y_train.sum() / len(y_train) * 100
test_fraud_rate  = y_test.sum()  / len(y_test)  * 100

print(f"Training set:   {X_train.shape[0]:,} rows | "
      f"Fraud: {y_train.sum()} ({train_fraud_rate:.4f}%)")
print(f"Test set:       {X_test.shape[0]:,} rows  | "
      f"Fraud: {y_test.sum()} ({test_fraud_rate:.4f}%)")
print(f"Features:       {X_train.shape[1]}")

Training set:   198,608 rows | Fraud: 331 (0.1667%)
Test set:       85,118 rows  | Fraud: 142 (0.1668%)
Features:       30


## Feature Scaling (StandardScaler)

### We apply StandardScaler to normalise all 30 features to the same magnitude range.

#### Why StandardScaler here instead of RobustScaler:
- RobustScaler was chosen previously because we were keeping outliers
  as fraud signal in a supervised model and didn't want them distorting
  the scale of normal transactions
- Now Isolation Forest is responsible for detecting outliers itself
- StandardScaler normalises using mean and standard deviation which
  gives Isolation Forest a clean, centred feature space to work in
- All features contribute equally to the isolation score Time's
  range of 0-172,792 no longer dominates over V1-V28

#### Critical rule  fit on training data only, transform both:
- scaler.fit_transform(X_train) learns mean and std from training,
  applies it to training
- scaler.transform(X_test) applies the SAME learned parameters to
  test set without refitting
- Refitting on test data would be data leakage



In [10]:
# Initialise StandardScaler 
scaler = StandardScaler()

# Fit on training data only, then transform 
# fit_transform learns the mean and std from X_train and applies
# the scaling in one step only ever called on training data
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns
)

# Transform test set using training statistics 
# transform applies the already-learned parameters from X_train
# the test set never influences the scaler's mean or std values
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns
)

# Verify scaling results 
print("Scaling complete.")
print()
print("Training set statistics after scaling:")
print(f"  Time   — mean: {X_train_scaled['Time'].mean():.4f}  "
      f"std: {X_train_scaled['Time'].std():.4f}")
print(f"  Amount — mean: {X_train_scaled['Amount'].mean():.4f}  "
      f"std: {X_train_scaled['Amount'].std():.4f}")
print(f"  V14    — mean: {X_train_scaled['V14'].mean():.4f}  "
      f"std: {X_train_scaled['V14'].std():.4f}")
print()
print(f"Training shape:  {X_train_scaled.shape}")
print(f"Test shape:      {X_test_scaled.shape}")

Scaling complete.

Training set statistics after scaling:
  Time   — mean: -0.0000  std: 1.0000
  Amount — mean: -0.0000  std: 1.0000
  V14    — mean: 0.0000  std: 1.0000

Training shape:  (198608, 30)
Test shape:      (85118, 30)


## Isolate Legitimate Transactions for Training

### This is the most important architectural decision in the entire pipeline.

### Isolation Forest is trained ONLY on legitimate transactions.

### Why:
#### 1. Isolation Forest learns what "normal" looks like by building random trees and measuring how many splits it takes to isolate each point
#### 2. Points that are easy to isolate (few splits needed) are anomalies
#### 3. Points that are hard to isolate (many splits needed) are normal,they blend in with the dense cluster of legitimate transactions
#### 4. If we trained on both legitimate and fraud, the algorithm would learn fraud patterns as part of "normal" and lose its ability to detect them as anomalies
#### 5. By training only on legitimate transactions, any fraud case thatarrives at inference time will be structurally different from everything the model has ever seen making it easy to isolate



In [11]:
# Extract only legitimate transactions from training set 
# We use the training labels (y_train) to filter X_train_scaled
# The test set is never touched — y_test remains sealed
X_train_legit = X_train_scaled[y_train.values == 0]

#  Confirm what the model will train on 
print("Isolation Forest training data:")
print(f"  Legitimate transactions: {len(X_train_legit):,}")
print(f"  Fraud transactions:      0 (intentionally excluded)")
print(f"  Features:                {X_train_legit.shape[1]}")
print()
print("These are the only transactions the model will ever see")
print("during training. Fraud patterns are completely unknown to it.")
print()

#  Quick sanity check: confirm zero fraud in training data 
# y_train.values == 0 creates a boolean mask
# X_train_scaled[mask] filters to only rows where mask is True
fraud_in_training = (y_train.values == 0).sum()
print(f"Sanity check — rows selected: {len(X_train_legit):,}")
print(f"Expected (legit in train):    "
      f"{(y_train.values == 0).sum():,}")
print(f"Match: {len(X_train_legit) == fraud_in_training}")

Isolation Forest training data:
  Legitimate transactions: 198,277
  Fraud transactions:      0 (intentionally excluded)
  Features:                30

These are the only transactions the model will ever see
during training. Fraud patterns are completely unknown to it.

Sanity check — rows selected: 198,277
Expected (legit in train):    198,277
Match: True


## Save Preprocessed Data and Scaler

#### We serialise the scaler and all prepared datasets to disk so Notebook 3 can load them directly without repeating any preprocessing steps.

#### Artefacts saved:
- scaler.joblib         — fitted StandardScaler (training stats only)
- X_train_legit.joblib  — 226,602 legitimate transactions for IF training
- X_test_scaled.joblib  — 56,746 scaled test transactions for evaluation
- y_test.joblib         — 56,746 true labels for evaluation
- X_train_scaled.joblib — full scaled training set (for leakage checks)
- y_train.joblib        — full training labels (for leakage checks)

In [12]:
import joblib
import os

# Create models directory if it doesn't exist 
os.makedirs('../models', exist_ok=True)

# Save the fitted scaler
# This scaler learned its parameters from X_train only
# It must be loaded at inference time to scale incoming transactions
# using the exact same mean and std values learned here
joblib.dump(scaler, '../models/scaler.joblib')

# Save the Isolation Forest training data 
# Only legitimate transactions — what IF will train on
joblib.dump(X_train_legit, '../models/X_train_legit.joblib')

# Save the test set 
# Sealed until evaluation in Notebook 3
joblib.dump(X_test_scaled, '../models/X_test_scaled.joblib')
joblib.dump(y_test,        '../models/y_test.joblib')

# Save full training set and labels 
# Used for leakage verification in Notebook 3
joblib.dump(X_train_scaled, '../models/X_train_scaled.joblib')
joblib.dump(y_train,        '../models/y_train.joblib')

# Confirm all files written correctly 
artefacts = [
    'scaler.joblib',
    'X_train_legit.joblib',
    'X_test_scaled.joblib',
    'y_test.joblib',
    'X_train_scaled.joblib',
    'y_train.joblib'
]

print("Preprocessing artefacts saved:")
print()
for f in artefacts:
    path    = f'../models/{f}'
    size_kb = os.path.getsize(path) / 1024
    print(f"  {f:<30} {size_kb:>8.1f} KB")

print()
print("Notebook 2 complete. Ready for Isolation Forest training.")

Preprocessing artefacts saved:

  scaler.joblib                       1.9 KB
  X_train_legit.joblib            48021.4 KB
  X_test_scaled.joblib            19950.7 KB
  y_test.joblib                    2660.7 KB
  X_train_scaled.joblib           46550.0 KB
  y_train.joblib                   6207.3 KB

Notebook 2 complete. Ready for Isolation Forest training.
